In [1]:
import sys
sys.path.append(r"C:\Users\baptiste.menetrier\Desktop\devPy\phd")

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

from propa.rtf.rtf_localisation.uace_testcase.src.antenna import SparseAntenna
from propa.rtf.rtf_localisation.uace_testcase.src.simulation import Simulation
from propa.rtf.rtf_localisation.uace_testcase.src.data_builder import DataBuilder
from propa.rtf.rtf_localisation.uace_testcase.src.feature_builder import FeatureBuilder
from propa.rtf.rtf_localisation.uace_testcase.src.localization_processor import LocalizationProcessor
from propa.rtf.rtf_localisation.uace_testcase.src.testcase_builder import (
    DeepWaterPekerisMunk,
    DeepWaterPekerisRhumrumSSP,
    DeepWaterRealEnv,
)

# Set array properties 

In [3]:
antenna = SparseAntenna(
    name="Test_sparse_antenna", n_elements=6, random_radius=5e3, rng_seed=42
)
antenna.plot_antenna()

# Set simulation properties

In [ ]:
debug = False
check = False
n_mc = 1
search_area_length = 1e3
use_weighted_rtf = False
# name = "wheighted_rtf"
name = "not_wheighted_rtf"
simu = Simulation(
    name=name,
    debug=debug,
    antenna=antenna,
    check_features=check,
    monte_carlo_iterations=n_mc,
    use_weighted_rtf=use_weighted_rtf,
    search_area_length=search_area_length,
)

# Build transfert function dataset 

In [ ]:
# db = DataBuilder(simulation=simu)
# db.build_tf_dataset()
# db.grid_dataset()
# db.build_signal()

# Derive features 

In [ ]:
# snr = 25
# fb = FeatureBuilder(simulation=simu)
# fb.build_features_from_time_signal(snr_dB=snr)
# fb.build_features_fullsimu()

# Process localization

In [ ]:
# snrs = [-10, -5, -2, -1, 0, 5, 10, 15, 20]
snrs = np.arange(-10, 17, 2)
# print(snrs)
lp = LocalizationProcessor(simulation=simu)
lp.process_multiple_snrs(snrs=snrs)

In [ ]:
debug = False
check = False
n_mc = 100
search_area_length = 1e3
use_weighted_rtf = True
name = "wheighted_rtf"
# name = "not_wheighted_rtf"
simu2 = Simulation(
    name=name,
    debug=debug,
    antenna=antenna,
    check_features=check,
    monte_carlo_iterations=n_mc,
    use_weighted_rtf=use_weighted_rtf,
    search_area_length=search_area_length,
)

In [ ]:
lp2 = LocalizationProcessor(simulation=simu2)
lp2.process_multiple_snrs(snrs=snrs)

In [ ]:
subarrays_list = np.atleast_2d(simu2.antenna.rcv_idx)
msr, dr, rmse = lp.load_msr_rmse_res_subarrays(
    subarrays_list=subarrays_list
)
msr2, dr2, rmse2 = lp2.load_msr_rmse_res_subarrays(
    subarrays_list=subarrays_list
)

In [ ]:

root_img = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\img\illustration\rtf\rtf_localisation\uace_testcase\comparaison_w"
for sa_key in msr.keys():

    # Extract info dataframes for current subarray
    rcv_ids = [f"{id[0]}_{id[1]}" for id in sa_key.split("_")]
    rcv_str = "$" + ", \,".join(rcv_ids) + "$"
    dr_sa = dr[sa_key]
    msr_sa = msr[sa_key]
    rmse_sa = rmse[sa_key]
    dr_sa2 = dr2[sa_key]
    msr_sa2 = msr2[sa_key]
    rmse_sa2 = rmse2[sa_key]

    # Plot msr
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.errorbar(
        msr_sa.index,
        msr_sa.dcf_mean,
        yerr=msr_sa.dcf_std,
        fmt="o-",
        label="DCF",
    )
    ax.errorbar(
        msr_sa.index,
        msr_sa.rtf_mean,
        yerr=msr_sa.rtf_std,
        fmt="o-",
        label="RTF",
    )
    ax.errorbar(
        msr_sa.index,
        msr_sa2.dcf_mean,
        yerr=msr_sa2.dcf_std,
        fmt="o-",
        label="DCF - wheighted",
    )
    ax.errorbar(
        msr_sa.index,
        msr_sa2.rtf_mean,
        yerr=msr_sa2.rtf_std,
        fmt="o-",
        label="RTF - wheighted",
    )
    ax.set_xlabel("SNR [dB]")
    ax.set_ylabel("MSR [dB]")
    ax.legend()
    ax.grid()

    plt.suptitle(f"Receivers = ({rcv_str})")

    fpath = os.path.join(root_img, f"msr_snr_{sa_key}.png")
    plt.savefig(fpath)
    # plt.close("all")

    # Plot dr
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.errorbar(
        dr_sa.index,
        dr_sa.dcf_mean,
        yerr=dr_sa.dcf_std,
        fmt="o-",
        label="DCF",
    )
    ax.errorbar(
        dr_sa.index,
        dr_sa.rtf_mean,
        yerr=dr_sa.rtf_std,
        fmt="o-",
        label="RTF",
    )
    ax.errorbar(
        dr_sa.index,
        dr_sa2.dcf_mean,
        yerr=dr_sa2.dcf_std,
        fmt="o-",
        label="DCF - wheigthed",
    )
    ax.errorbar(
        dr_sa.index,
        dr_sa2.rtf_mean,
        yerr=dr_sa2.rtf_std,
        fmt="o-",
        label="RTF - wheigthed",
    )
    plt.suptitle(f"Receivers = ({rcv_str})")
    ax.set_ylabel(r"$\Delta_r$" + " [m]")
    ax.set_xlabel("SNR [dB]")
    ax.legend()
    ax.grid()
    fpath = os.path.join(root_img, f"dr_pos_snr_{sa_key}.png")
    plt.savefig(fpath)

    # Plot rmse
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.plot(rmse_sa.index, rmse_sa["dcf"], "o-", label="DCF")
    ax.plot(rmse_sa.index, rmse_sa["rtf"], "o-", label="RTF")
    ax.plot(rmse_sa.index, rmse_sa2["dcf"], "o-", label="DCF - wheighted")
    ax.plot(rmse_sa.index, rmse_sa2["rtf"], "o-", label="RTF - wheighted")
    plt.suptitle(f"Receivers = ({rcv_str})")
    ax.set_xlabel("SNR [dB]")
    ax.set_ylabel("RMSE [m]")
    ax.legend()
    ax.grid()

    fpath = os.path.join(root_img, f"rmse_pos_snr_{sa_key}.png")
    plt.savefig(fpath)

# Munk ssp profile testcase 
Deep water testcase with Munk ssp profile 

## Init testcase 

In [ ]:
debug = False
check = True
n_mc = 1
use_weighted_rtf = True
name = "dw_pekeris_munk"

search_area_length = 2*1e3
simu = Simulation(
    name=name,
    debug=debug,
    antenna=antenna,
    check_features=check,
    monte_carlo_iterations=n_mc,
    use_weighted_rtf=use_weighted_rtf,
    search_area_length=search_area_length,
)
test_case = DeepWaterPekerisMunk(simulation=simu, mode="run")

In [ ]:
print(f"Position of the event source:  (x = {simu.event_ship_x:.2f} m, y = {simu.event_ship_y:.2f} m)")

## Build dataset


In [ ]:
db = DataBuilder(simulation=simu)
db.build_tf_dataset()
db.grid_dataset()
db.build_signal()

## Process localization

In [ ]:
# snrs = np.arange(-10, 17, 2)
snrs = [5]
lp_munk = LocalizationProcessor(simulation=simu)
lp_munk.process_multiple_snrs(snrs=snrs)

# Deep water with real ssp profile from rhumrum area (RR48)

In [ ]:
debug = False
check = True
n_mc = 1
use_weighted_rtf = True
name = "dw_pekeris_rhumrum_ssp"

search_area_length = 1 * 1e3
simu = Simulation(
    name=name,
    debug=debug,
    antenna=antenna,
    check_features=check,
    monte_carlo_iterations=n_mc,
    use_weighted_rtf=use_weighted_rtf,
    search_area_length=search_area_length,
)
test_case = DeepWaterPekerisRhumrumSSP(simulation=simu, mode="run")

## Build dataset


In [ ]:
db = DataBuilder(simulation=simu)
db.build_tf_dataset()
db.grid_dataset()
db.build_signal()

## Process localization

In [ ]:
# snrs = np.arange(-10, 17, 2)
snrs = [5]
lp_munk = LocalizationProcessor(simulation=simu)
lp_munk.process_multiple_snrs(snrs=snrs)

# Real deep water with real ssp profile and real bathy from rhumrum area (RR48)

In [4]:
debug = False
check = True
n_mc = 1
use_weighted_rtf = True
name = "dw_real_env"

search_area_length = 1 * 1e3
simu = Simulation(
    name=name,
    debug=debug,
    antenna=antenna,
    check_features=check,
    monte_carlo_iterations=n_mc,
    use_weighted_rtf=use_weighted_rtf,
    search_area_length=search_area_length,
)
test_case = DeepWaterRealEnv(simulation=simu, mode="run")

C:\Users\baptiste.menetrier\Desktop\devPy\phd\propa\kraken_toolbox\src\kraken_env.py:199: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
C:\Users\baptiste.menetrier\Desktop\devPy\phd\propa\kraken_toolbox\src\kraken_testcase.py:316: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


## Build dataset


In [5]:
db = DataBuilder(simulation=simu)
db.build_tf_dataset()
db.grid_dataset()
db.build_signal()

## Process localization

In [7]:
# snrs = np.arange(-10, 17, 2)
snrs = [50]
lp_munk = LocalizationProcessor(simulation=simu)
lp_munk.process_multiple_snrs(snrs=snrs)

c:\Users\baptiste.menetrier\.virtualenvs\phd-FBjz4hBd\Lib\site-packages\scipy\signal\_spectral_py.py:1240: UserWarning: nperseg = 2048 is greater than input length  = 2000, using nperseg = 2000
  freqs, time, Zxx = _spectral_helper(x, x, fs, window, nperseg, noverlap,
c:\Users\baptiste.menetrier\.virtualenvs\phd-FBjz4hBd\Lib\site-packages\scipy\signal\_spectral_py.py:600: UserWarning: nperseg = 2048 is greater than input length  = 2000, using nperseg = 2000
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


Features derived from time signal in 91.99 s
